# DATA 298B LLM Training Orchestrator

This notebook is an orchestration layer only. QLoRA/SFT remains in `src.training.qlora_sft`; the notebook configures the run, persists artifacts, validates the environment, and performs post-training checks.

In [ ]:
# ---- Edit only this cell for a new experiment ----
REPO_URL = 'https://github.com/aryaMehta26/Graph-Based-Fraud-Intelligence-Platform.git'
REPO_BRANCH = 'feature/298b-backend'
MODEL = 'ministral'          # ministral | gemma | granite | qwen
MODE = 'pilot'               # pilot | full
MOUNT_DRIVE = True
RUN_DRY_RUN_VALIDATION = False
HF_LOGIN_REQUIRED = True
PUSH_PRIVATE_ADAPTER = False
HF_ADAPTER_REPO = ''        # e.g. your-org/aml-ministral- pilot
PILOT_CASES = 100
PILOT_STEPS = 75
FULL_STEPS = 1000
SEED = 42

assert MODEL in {'ministral', 'gemma', 'granite', 'qwen'}
assert MODE in {'pilot', 'full'}
if PUSH_PRIVATE_ADAPTER: assert HF_ADAPTER_REPO, 'Set HF_ADAPTER_REPO before enabling Hub upload'
print({'MODEL': MODEL, 'MODE': MODE, 'REPO_BRANCH': REPO_BRANCH})

In [ ]:
# Clone or update the selected feature branch. No credentials are stored here.
from pathlib import Path
import os, subprocess
REPO_DIR = Path('/content/Graph-Based-Fraud-Intelligence-Platform')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('repository:', REPO_DIR, 'branch:', subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip())

In [ ]:
# Optional persistent storage. Artifacts are written under Drive when mounted.
DRIVE_ROOT = Path('/content/drive/MyDrive/DATA298B')
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
else:
    DRIVE_ROOT = REPO_DIR / 'artifacts' / 'colab_local'
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR = DRIVE_ROOT / 'adapters' / f'{MODEL}_{MODE}'
CHECKPOINT_DIR = ADAPTER_DIR / 'checkpoints'
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
print('persistent artifact root:', DRIVE_ROOT)

In [ ]:
# Install the repository requirements plus the GPU training/runtime stack.
%pip install -q -r requirements.txt
%pip install -q accelerate bitsandbytes sentencepiece huggingface_hub
print('dependencies installed')

In [ ]:
# Secure Hugging Face login: Colab prompts for a token or reads a Colab Secret named HF_TOKEN.
if HF_LOGIN_REQUIRED:
    from huggingface_hub import login
    token = None
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
    except Exception:
        pass
    login(token=token or None, add_to_git_credential=False)
    print('Hugging Face authentication configured without storing a token in notebook source.')

In [ ]:
# Fail early before downloading a 14B--31B model if the GPU is clearly undersized.
import importlib.metadata as md
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU is required. Enable a Colab GPU runtime before starting training.')
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
required_gb = 20 if MODEL == 'ministral' else 28
versions = {name: (md.version(name) if importlib.util.find_spec(name) else 'not installed') for name in ['torch', 'transformers', 'bitsandbytes']}
print('GPU:', gpu_name)
print(f'VRAM: {vram_gb:.1f} GiB; required early-stop threshold: {required_gb} GiB')
print('CUDA:', torch.version.cuda)
print('versions:', versions)
if vram_gb < required_gb:
    raise RuntimeError(f'Insufficient GPU memory for {MODEL}: {vram_gb:.1f} GiB available, approximately {required_gb} GiB required. Choose a higher-memory runtime or a smaller experiment.')

In [ ]:
# Verify the registry entry and select the existing SFT JSONL dataset.
import yaml
registry_path = REPO_DIR / 'configs/models/registry.yaml'
registry = yaml.safe_load(registry_path.read_text())
if MODEL not in registry:
    raise KeyError(f'{MODEL!r} is not in {registry_path}; available: {sorted(registry)}')
SPEC = registry[MODEL]
print('selected:', SPEC)

# Prefer a persistent Drive dataset; fall back to the repository's existing pilot SFT JSONL.
DRIVE_DATASET = DRIVE_ROOT / 'datasets' / 'aml_sft.jsonl'
REPO_DATASET = REPO_DIR / 'artifacts/evaluation_dry_run/sft.jsonl'
DATASET = DRIVE_DATASET if DRIVE_DATASET.exists() else REPO_DATASET
if not DATASET.exists():
    raise FileNotFoundError(f'No SFT JSONL found. Put the existing dataset at {DRIVE_DATASET} or run the optional dry-run after supplying processed project data.')
print('dataset:', DATASET, 'bytes:', DATASET.stat().st_size)

In [ ]:
# Optional repository validation. This is skipped by default to avoid rescanning large local Parquets.
if RUN_DRY_RUN_VALIDATION:
    subprocess.run(['python', 'scripts/run_298b_dry_run.py'], check=True)
else:
    print('dry run skipped; set RUN_DRY_RUN_VALIDATION=True to execute it')

In [ ]:
# Create a pilot-only view without changing the canonical dataset.
import json
TRAIN_DATASET = DATASET
if MODE == 'pilot':
    PILOT_DATASET = DRIVE_ROOT / 'datasets' / f'aml_sft_{PILOT_CASES}.jsonl'
    with DATASET.open() as src, PILOT_DATASET.open('w') as dst:
        for index, line in enumerate(src):
            if index >= PILOT_CASES: break
            if line.strip(): dst.write(line)
    TRAIN_DATASET = PILOT_DATASET
STEPS = PILOT_STEPS if MODE == 'pilot' else FULL_STEPS
print('training dataset:', TRAIN_DATASET, 'steps:', STEPS)

In [ ]:
# Call the existing trainer. No QLoRA implementation is duplicated here.
from src.training.qlora_sft import train
metrics = train(MODEL, TRAIN_DATASET, ADAPTER_DIR, max_steps=STEPS, max_samples=PILOT_CASES if MODE == 'pilot' else None, seed=SEED)
print('training metrics:', metrics)
print('adapter directory:', ADAPTER_DIR)

In [ ]:
# Verify adapter files, tokenizer/configuration, metadata, and loss history.
from pathlib import Path
required = ['adapter_config.json', 'tokenizer_config.json', 'training_run.json', 'loss_history.json']
missing = [name for name in required if not (ADAPTER_DIR / name).exists()]
if not any((ADAPTER_DIR / name).exists() for name in ['adapter_model.safetensors', 'adapter_model.bin']): missing.append('adapter_model.safetensors or adapter_model.bin')
if missing:
    raise FileNotFoundError(f'Training completed but expected artifacts are missing: {missing}')
print('saved artifacts:')
for path in sorted(ADAPTER_DIR.rglob('*')):
    if path.is_file(): print(' -', path)

# Load the adapter configuration without downloading a second full model.
from peft import PeftConfig
adapter_config = PeftConfig.from_pretrained(str(ADAPTER_DIR))
print('adapter loads; base model:', adapter_config.base_model_name_or_path)

In [ ]:
# One post-training AML inference example through the shared backend/agent interfaces.
# This reloads the quantized base model plus adapter and therefore needs sufficient VRAM.
from src.aml.backend import TransformersBackend
from src.aml.agent import AMLAgent
from src.aml.schemas import InvestigationCase
case_payload = json.loads(next(TRAIN_DATASET.open()))
case = InvestigationCase(case_id=case_payload['case_id'], context=json.loads(case_payload['messages'][1]['content']))
backend = TransformersBackend(SPEC['model_id'], revision=SPEC['revision'], adapter_path=str(ADAPTER_DIR), model_kwargs={'load_in_4bit': True, 'device_map': 'auto'})
trace = AMLAgent(backend, max_tool_calls=3, artifact_dir=str(ADAPTER_DIR / 'inference_traces')).investigate(case, model_family=MODEL, variant='finetuned', seed=SEED)
print(json.dumps(trace['report'], indent=2, default=str))

In [ ]:
# Optional private Hub upload. The token is held by the Hugging Face client, never written here.
if PUSH_PRIVATE_ADAPTER:
    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(repo_id=HF_ADAPTER_REPO, private=True, exist_ok=True)
    api.upload_folder(folder_path=str(ADAPTER_DIR), repo_id=HF_ADAPTER_REPO, repo_type='model')
    print('private adapter uploaded:', f'https://huggingface.co/{HF_ADAPTER_REPO}')
else:
    print('Hub upload skipped; adapter remains on Drive')

## Handoff

Final adapter path: `ADAPTER_DIR` printed above. In VS Code, pull/download that directory and load it with the shared backend:

```python
from src.aml.backend import TransformersBackend
backend = TransformersBackend(
    model_id_from_registry,
    revision=revision_from_registry,
    adapter_path='/path/to/adapter_directory',
    device_map='auto',
    torch_dtype='auto',
)
```

Then run the existing base-vs-fine-tuned benchmark/evaluation workflow. To train another family, change only `MODEL` in the first code cell.